In [35]:
import pandas as pd
import numpy as np
import re

df = pd.read_csv("../data/clean_products.csv")

df.shape

(1666, 24)

In [36]:
def clean_text(text):

    text = str(text).lower()

    text = re.sub(r'[^a-z0-9\s]', ' ', text)

    text = re.sub(r'\s+', ' ', text)

    return text.strip()

In [37]:
df["weighted_search_text"] = (
    df["title"].fillna("") + " " +
    df["title"].fillna("") + " " +
    df["title"].fillna("") + " " +
    df["brand"].fillna("") + " " +
    df["category"].fillna("") + " " +
    df["specifications"].fillna("") + " " +
    df["description"].fillna("")
)

In [38]:
df["weighted_search_text"] = (
    df["weighted_search_text"]
    .astype(str)
    .apply(clean_text)
)

In [39]:
df["weighted_search_text"].isnull().sum()

np.int64(0)

In [40]:
df.loc[27, "weighted_search_text"]

'apple iphone 11 128gb non pta apple iphone 11 128gb non pta apple iphone 11 128gb non pta apple mobile ram 1 memory quantity 4gb internal storage space 128gb main camera pixels 12mp 12mp battery capacity 3110 mah screen size 6 1 inches 5g support no finger print no display technology liquid retina ips lcd capacitive touchscreen 16m colors multitouch display ips lcd number of colours 16m colors scratch resistant display screen resolution 828 x 1792 pixels pixel density 324 ppi dual screens sd card no built in camera yes auto focus yes built in flash yes digital zoom still image f 1 8 maximum resolution still 2160p number of cameras 3pc front camera resolution 12mp optical zoom yes video recorder yes digital zoom video yes maximum numbers of fps when recording 24 30 60fps maximum resolution video 1080p graphics processor yes graphics processor type apple a13 7 nm processor core type processor speed chipset cpu type 4g lte yes bluetooth yes nfc support yes wifi wireless fidelity yes colo

In [43]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(
    stop_words="english"
)

tfidf_matrix = vectorizer.fit_transform(
    df["weighted_search_text"]
)

In [44]:
from sklearn.metrics.pairwise import cosine_similarity

def search_products(query, top_n=10):

    query = clean_text(query)

    query_vector = vectorizer.transform([query])

    similarity_scores = cosine_similarity(
        query_vector,
        tfidf_matrix
    )

    scores = similarity_scores.flatten()

    top_indices = scores.argsort()[-top_n:][::-1]

    results = df.iloc[top_indices][
        ["title", "brand", "category"]
    ].copy()

    results["score"] = scores[top_indices]

    return results

In [47]:
search_products("iphone 128gb")

,title,brand,category,score
27,Apple iPhone 11 128GB NON PTA,Apple,Mobile,0.334649
21,Apple iPhone 11 128GB PTA Approved,Apple,Mobile,0.330491
20,APPLE IPHONE 13 4GB RAM 128GB STORAGE SINGLE S...,Apple,Mobile,0.278446
84,Apple iPhone 14 Pro Max 128GB Storage Dual Sim...,Apple,Mobile,0.273889
108,Apple iPhone 14 Pro Max 128GB Physical Sim Non...,Apple,Mobile,0.273159
117,Apple iPhone 14 Pro 128GB Storage Dual Sim Non...,Apple,Mobile,0.272457
88,Apple iPhone 14 Pro Max 128GB Storage PTA Appr...,Apple,Mobile,0.270607
92,Apple iPhone 14 Pro 128GB Storage PTA Approved...,Apple,Mobile,0.268431
105,Apple iPhone 14 Pro 128GB Storage Physical Sim...,Apple,Mobile,0.266257
58,Apple iPhone 14 Pro Max 128GB Storage Dual Sim...,Apple,Mobile,0.263520


In [46]:
search_products("samsung")


,title,brand,category,score
496,"Samsung Galaxy Tab S6 Lite 10.4""",Samsung,Mobile,0.411390
498,"Samsung Galaxy Tab S6 Lite 10.4""",Samsung,Mobile,0.411101
804,Samsung Galaxy Buds 2,NaN,Earbuds,0.391443
490,Samsung Galaxy Tab A7 Lite,Samsung,Mobile,0.373616
815,Samsung Galaxy Buds 2 Pro,NaN,Earbuds,0.371304
812,Samsung Galaxy Buds Pro,NaN,Earbuds,0.367236
516,"Samsung Galaxy Tab S8 11"" 128GB | Czone.com.pk",Samsung,Mobile,0.361062
497,"Samsung Galaxy Tab S6 Lite 10.4"", Wi-Fi Only, ...",Samsung,Mobile,0.346151
1258,Samsung Galaxy Fit 2,NaN,Watch,0.338726
819,Samsung Buds Live,NaN,Earbuds,0.326229


In [48]:
search_products("8gb ram mobile")

,title,brand,category,score
1617,me Mobile Power Max,NaN,Mobile,0.207170
0,Nothing Phone 1 8GB RAM 256GB Storage Non PTA ...,NaN,Mobile,0.196516
1419,me Mobile Magic Max,NaN,Mobile,0.186831
1628,me Mobile Power Music,NaN,Mobile,0.179752
1507,me Mobile Magic Sound,NaN,Mobile,0.177277
1610,me Mobile Power Bold,NaN,Mobile,0.176421
515,"Samsung Galaxy Tab S8 11"" 128GB SSD, 8GB RAM, ...",Samsung,Mobile,0.176237
1611,me Mobile L786,NaN,Mobile,0.176200
527,Microsoft Surface Pro 9 Tablet Laptop - Intel ...,Microsoft,Mobile,0.176135
1599,me Mobile Power Star,NaN,Mobile,0.174940


In [49]:
df[
    [
        "title",
        "average_rating",
        "num_ratings"
    ]
].head(10)

,title,average_rating,num_ratings
0,Nothing Phone 1 8GB RAM 256GB Storage Non PTA ...,0.0,0.0
1,Oppo F21 Pro 8GB Ram 128GB Storage 5G PTA Appr...,0.0,0.0
2,Tecno Spark 10,0.0,0.0
3,Vivo V27 5G,0.0,0.0
4,Apple Iphone 15 Pro Max,0.0,0.0
5,Realme GT3,0.0,0.0
6,Sparx S9 2GB RAM 32GB Storage PTA Approved,0.0,0.0
7,Sparx S6 2GB RAM 32GB Storage,0.0,0.0
8,Tecno Pova Neo 2 4GB RAM 64GB Storage PTA Appr...,0.0,0.0
9,Vivo Y73 8GB RAM 128GB Storage PTA Approved,0.0,0.0


In [52]:
df["average_rating"].describe()

count    1666.000000
mean        2.696879
std         2.489790
min         0.000000
25%         0.000000
50%         5.000000
75%         5.000000
max         5.000000
Name: average_rating, dtype: float64

In [53]:
df["num_ratings"].describe()

count    1666.000000
mean        7.201681
std        32.431959
min         0.000000
25%         0.000000
50%         1.000000
75%         1.000000
max       751.000000
Name: num_ratings, dtype: float64

In [54]:
(df["average_rating"] > 0).sum()

np.int64(900)

In [55]:
(df["num_ratings"] > 0).sum()

np.int64(900)

In [56]:
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()

df["rating_score"] = scaler.fit_transform(
    df[["average_rating"]]
)

In [57]:
df["num_ratings"].describe()

count    1666.000000
mean        7.201681
std        32.431959
min         0.000000
25%         0.000000
50%         1.000000
75%         1.000000
max       751.000000
Name: num_ratings, dtype: float64

In [58]:
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()

df["rating_score"] = scaler.fit_transform(
    df[["average_rating"]]
)

In [60]:
df[
    df["average_rating"] > 0
][
    [
        "average_rating",
        "rating_score"
    ]
].head(10)

,average_rating,rating_score
397,5.0,1.0
402,5.0,1.0
406,3.5,0.7
408,5.0,1.0
409,4.0,0.8
410,3.5,0.7
412,5.0,1.0
464,5.0,1.0
774,5.0,1.0
775,5.0,1.0


In [61]:
df["rating_score"].describe()

count    1666.000000
mean        0.539376
std         0.497958
min         0.000000
25%         0.000000
50%         1.000000
75%         1.000000
max         1.000000
Name: rating_score, dtype: float64

In [62]:
import numpy as np

df["popularity_score"] = np.log1p(
    df["num_ratings"]
)

In [63]:
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()

df["popularity_score"] = scaler.fit_transform(
    df[["popularity_score"]]
)

In [64]:
df[
    [
        "num_ratings",
        "popularity_score"
    ]
].sample(10)

,num_ratings,popularity_score
97,0.0,0.000000
107,0.0,0.000000
1194,4.0,0.243017
1343,16.0,0.427801
1339,1.0,0.104662
980,1.0,0.104662
261,0.0,0.000000
1557,6.0,0.293823
836,4.0,0.243017
546,0.0,0.000000


In [75]:
from sklearn.metrics.pairwise import cosine_similarity

def search_products_v2(query, top_n=10):

    query = clean_text(query)

    query_vector = vectorizer.transform([query])

    similarity_scores = cosine_similarity(
        query_vector,
        tfidf_matrix
    ).flatten()

    temp_df = df.copy()

    temp_df["similarity_score"] = similarity_scores

    candidate_df = temp_df.nlargest(
        100,
        "similarity_score"
    )

    candidate_df["final_score"] = (
        0.90 * candidate_df["similarity_score"]
        + 0.07 * candidate_df["rating_score"]
        + 0.03 * candidate_df["popularity_score"]
    )

    results = candidate_df.sort_values(
        "final_score",
        ascending=False
    )
    return results[
        [
            "title",
            "brand",
            "average_rating",
            "num_ratings",
            "similarity_score",
            "final_score"
        ]
    ].head(top_n)

In [76]:
search_products_v2("iphone 128gb")

,title,brand,average_rating,num_ratings,similarity_score,final_score
27,Apple iPhone 11 128GB NON PTA,Apple,0.0,0.0,0.334649,0.301184
21,Apple iPhone 11 128GB PTA Approved,Apple,0.0,0.0,0.330491,0.297442
1592,Apple iPhone 13,NaN,5.0,6.0,0.221811,0.278444
1564,Apple iPhone 11,NaN,5.0,127.0,0.195527,0.267953
20,APPLE IPHONE 13 4GB RAM 128GB STORAGE SINGLE S...,Apple,0.0,0.0,0.278446,0.250601
84,Apple iPhone 14 Pro Max 128GB Storage Dual Sim...,Apple,0.0,0.0,0.273889,0.246500
108,Apple iPhone 14 Pro Max 128GB Physical Sim Non...,Apple,0.0,0.0,0.273159,0.245844
117,Apple iPhone 14 Pro 128GB Storage Dual Sim Non...,Apple,0.0,0.0,0.272457,0.245211
88,Apple iPhone 14 Pro Max 128GB Storage PTA Appr...,Apple,0.0,0.0,0.270607,0.243546
92,Apple iPhone 14 Pro 128GB Storage PTA Approved...,Apple,0.0,0.0,0.268431,0.241588


In [67]:
search_products_v2("samsung")

,title,brand,average_rating,num_ratings,similarity_score,final_score
804,Samsung Galaxy Buds 2,NaN,5.0,83.0,0.391443,0.510485
815,Samsung Galaxy Buds 2 Pro,NaN,5.0,37.0,0.371304,0.483404
812,Samsung Galaxy Buds Pro,NaN,5.0,32.0,0.367236,0.478222
1582,Samsung Galaxy A12,NaN,5.0,319.0,0.279253,0.446538
1571,Samsung Galaxy A32,NaN,5.0,751.0,0.255940,0.441955
1258,Samsung Galaxy Fit 2,NaN,5.0,11.0,0.338726,0.441566
1344,Samsung Galaxy A13,NaN,5.0,229.0,0.278730,0.441159
1587,Samsung Galaxy A03,NaN,5.0,177.0,0.282273,0.439947
819,Samsung Buds Live,NaN,5.0,13.0,0.326229,0.434520
1576,Samsung Galaxy A22,NaN,5.0,73.0,0.282821,0.427105


In [78]:
import os
print(os.getcwd())

c:\Users\BUNNY\OneDrive\Desktop\AI\product-search-ranking\notebooks


In [79]:
import sys
import os

project_root = os.path.abspath("..")
sys.path.append(project_root)

print(project_root)

c:\Users\BUNNY\OneDrive\Desktop\AI\product-search-ranking


In [80]:
from src.preprocessing import clean_text

clean_text("Apple iPhone 11!!! 128GB@@@")

'apple iphone 11 128gb'

In [81]:
from src.search_engine import search_products_v2

In [82]:
search_products_v2(
    "iphone 128gb",
    vectorizer,
    tfidf_matrix,
    df,
    clean_text
)

,title,brand,average_rating,num_ratings,similarity_score,final_score
27,Apple iPhone 11 128GB NON PTA,Apple,0.0,0.0,0.334649,0.301184
21,Apple iPhone 11 128GB PTA Approved,Apple,0.0,0.0,0.330491,0.297442
1592,Apple iPhone 13,NaN,5.0,6.0,0.221811,0.278444
1564,Apple iPhone 11,NaN,5.0,127.0,0.195527,0.267953
20,APPLE IPHONE 13 4GB RAM 128GB STORAGE SINGLE S...,Apple,0.0,0.0,0.278446,0.250601
84,Apple iPhone 14 Pro Max 128GB Storage Dual Sim...,Apple,0.0,0.0,0.273889,0.246500
108,Apple iPhone 14 Pro Max 128GB Physical Sim Non...,Apple,0.0,0.0,0.273159,0.245844
117,Apple iPhone 14 Pro 128GB Storage Dual Sim Non...,Apple,0.0,0.0,0.272457,0.245211
88,Apple iPhone 14 Pro Max 128GB Storage PTA Appr...,Apple,0.0,0.0,0.270607,0.243546
92,Apple iPhone 14 Pro 128GB Storage PTA Approved...,Apple,0.0,0.0,0.268431,0.241588


In [83]:
from src.ranking import create_ranking_features

df = create_ranking_features(df)

In [85]:
df[
    [
        "average_rating",
        "rating_score",
        "num_ratings",
        "popularity_score"
    ]
].sample(10)

,average_rating,rating_score,num_ratings,popularity_score
1420,5.0,1.0,1.0,0.104662
345,0.0,0.0,0.0,0.000000
888,5.0,1.0,1.0,0.104662
930,5.0,1.0,3.0,0.209324
407,0.0,0.0,0.0,0.000000
564,0.0,0.0,0.0,0.000000
628,0.0,0.0,0.0,0.000000
1042,5.0,1.0,1.0,0.104662
1560,5.0,1.0,1.0,0.104662
948,5.0,1.0,3.0,0.209324


In [86]:
print(df["rating_score"].min())
print(df["rating_score"].max())

print(df["popularity_score"].min())
print(df["popularity_score"].max())

0.0
1.0
0.0
1.0


In [87]:
result = search_products_v2(
    "iphone 128gb",
    vectorizer,
    tfidf_matrix,
    df,
    clean_text
)

print(type(result))

<class 'pandas.core.frame.DataFrame'>


In [88]:
import importlib
import src.search_engine

importlib.reload(src.search_engine)

<module 'src.search_engine' from 'c:\\Users\\BUNNY\\OneDrive\\Desktop\\AI\\product-search-ranking\\src\\search_engine.py'>

In [89]:
from src.search_engine import search_products_v2

In [90]:
result = search_products_v2(
    "iphone 128gb",
    vectorizer,
    tfidf_matrix,
    df,
    clean_text
)

print(type(result))

<class 'list'>


In [91]:
result[:2]

[{'title': 'Apple iPhone 11 128GB NON PTA ',
  'brand': 'Apple',
  'average_rating': 0.0,
  'num_ratings': 0.0,
  'similarity_score': 0.3346489853271246,
  'final_score': 0.30118408679441216},
 {'title': 'Apple iPhone 11 128GB PTA Approved ',
  'brand': 'Apple',
  'average_rating': 0.0,
  'num_ratings': 0.0,
  'similarity_score': 0.330491079504462,
  'final_score': 0.2974419715540158}]

In [92]:
print(type(result))

<class 'list'>


In [93]:
print(df.shape)
print(type(vectorizer))
print(type(tfidf_matrix))

(1666, 27)
<class 'sklearn.feature_extraction.text.TfidfVectorizer'>
<class 'scipy.sparse._csr.csr_matrix'>


In [94]:
import os

print(os.getcwd())

c:\Users\BUNNY\OneDrive\Desktop\AI\product-search-ranking\notebooks


In [95]:
import os

models_path = os.path.abspath("../models")

print(models_path)
print(os.path.exists(models_path))

c:\Users\BUNNY\OneDrive\Desktop\AI\product-search-ranking\models
True


In [96]:
import pickle
import os

products_path = os.path.abspath("../models/products.pkl")

print(products_path)

with open(products_path, "wb") as f:
    pickle.dump(df, f)

print("Saved!")
print("Size:", os.path.getsize(products_path))

c:\Users\BUNNY\OneDrive\Desktop\AI\product-search-ranking\models\products.pkl
Saved!
Size: 6713429


In [97]:
with open(products_path, "rb") as f:
    test_df = pickle.load(f)

print(test_df.shape)

(1666, 27)


In [100]:
import pickle
import os

vectorizer_path = os.path.abspath("../models/vectorizer.pkl")

with open(vectorizer_path, "wb") as f:
    pickle.dump(vectorizer, f)

print("Size:", os.path.getsize(vectorizer_path))

Size: 87494


In [101]:
tfidf_path = os.path.abspath("../models/tfidf_matrix.pkl")

with open(tfidf_path, "wb") as f:
    pickle.dump(tfidf_matrix, f)

print("Size:", os.path.getsize(tfidf_path))

Size: 1513499


In [102]:
with open(vectorizer_path, "rb") as f:
    test_vectorizer = pickle.load(f)

print(type(test_vectorizer))

<class 'sklearn.feature_extraction.text.TfidfVectorizer'>


In [103]:
with open(tfidf_path, "rb") as f:
    test_matrix = pickle.load(f)

print(type(test_matrix))
print(test_matrix.shape)

<class 'scipy.sparse._csr.csr_matrix'>
(1666, 4513)


In [104]:
import pickle

with open("../models/products.pkl", "rb") as f:
    test_df = pickle.load(f)

print(test_df.shape)

(1666, 27)


In [105]:
import sys
print(sys.executable)

c:\Users\BUNNY\OneDrive\Desktop\AI\product-search-ranking\venv\Scripts\python.exe


In [106]:
print(df.columns.tolist())

['id', 'slug', 'title', 'imgs', 'brand', 'category', 'vendor', 'used', 'address', 'availability', 'currency', 'original_price', 'discounted_price', 'specifications', 'description', 'delivery_fee', 'delivery_details', 'warranty', 'warranty_type', 'average_rating', 'num_ratings', 'reviews', 'search_text', 'clean_search_text', 'weighted_search_text', 'rating_score', 'popularity_score']


In [108]:
result = search_products_v2(
    "iphone 128gb",
    vectorizer,
    tfidf_matrix,
    df,
    clean_text
)

result

[{'title': 'Apple iPhone 11 128GB NON PTA ',
  'brand': 'Apple',
  'average_rating': 0.0,
  'num_ratings': 0.0,
  'similarity_score': 0.3346489853271246,
  'final_score': 0.30118408679441216},
 {'title': 'Apple iPhone 11 128GB PTA Approved ',
  'brand': 'Apple',
  'average_rating': 0.0,
  'num_ratings': 0.0,
  'similarity_score': 0.330491079504462,
  'final_score': 0.2974419715540158},
 {'title': 'Apple iPhone 13',
  'brand': nan,
  'average_rating': 5.0,
  'num_ratings': 6.0,
  'similarity_score': 0.2218107626142913,
  'final_score': 0.27844436681364837},
 {'title': 'Apple iPhone 11',
  'brand': nan,
  'average_rating': 5.0,
  'num_ratings': 127.0,
  'similarity_score': 0.19552715556802128,
  'final_score': 0.26795340773656795},
 {'title': 'APPLE IPHONE 13 4GB RAM 128GB STORAGE SINGLE SIM PTA Approved ',
  'brand': 'Apple',
  'average_rating': 0.0,
  'num_ratings': 0.0,
  'similarity_score': 0.27844556993030667,
  'final_score': 0.250601012937276},
 {'title': 'Apple iPhone 14 Pro Max 

In [109]:
result[:3]

[{'title': 'Apple iPhone 11 128GB NON PTA ',
  'brand': 'Apple',
  'average_rating': 0.0,
  'num_ratings': 0.0,
  'similarity_score': 0.3346489853271246,
  'final_score': 0.30118408679441216},
 {'title': 'Apple iPhone 11 128GB PTA Approved ',
  'brand': 'Apple',
  'average_rating': 0.0,
  'num_ratings': 0.0,
  'similarity_score': 0.330491079504462,
  'final_score': 0.2974419715540158},
 {'title': 'Apple iPhone 13',
  'brand': nan,
  'average_rating': 5.0,
  'num_ratings': 6.0,
  'similarity_score': 0.2218107626142913,
  'final_score': 0.27844436681364837}]

In [110]:
import math

result = search_products_v2(
    "iphone 128gb",
    vectorizer,
    tfidf_matrix,
    df,
    clean_text
)

for i, row in enumerate(result):
    for key, value in row.items():

        if isinstance(value, float):

            if math.isnan(value):
                print("NaN Found")
                print("Row:", i)
                print("Column:", key)

            if math.isinf(value):
                print("Infinity Found")
                print("Row:", i)
                print("Column:", key)

NaN Found
Row: 2
Column: brand
NaN Found
Row: 3
Column: brand
